Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates how to:
- create hedges and hedged liability portfolios based on original liability position

In [1]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\example_notebooks
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [2]:
# imports 
from typing import Dict
from datetime import datetime 
import numpy as np
import pandas as pd 
from rateslib import IRS as rlIRS, dcf as rlDCF 
from curve_engine.swap_curve_solver import SwapCurveSolver
from curve_engine.swap_curve_builder import SwapCurveBuilder
from curve_engine.swap_rate_pca_estimator import SwapRatePCAEstimator
from curve_engine.swap_rate_mc_simulator import SwapRateMCSimulator
from curve_engine.swap_curve_scenario_generator import SwapCurveScenarioGenerator
from position_engine.euribor6m_irs_position import Euribor6mIRSPosition
from position_engine.liability_position import LiabilityPosition 
from position_engine.liability_portfolio import LiabilityPortfolio
from hedge_engine.liability_hedge_constructor import LiabilityHedgeConstructor 

C:\Users\mduba\AppData\Local\Temp\ipykernel_23716\76073408.py:6: LicenceNotice: 
Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence
  from rateslib import IRS as rlIRS, dcf as rlDCF


In [3]:
# parameters 
curve_date = datetime(2026,3,31)

swap_rates =  {
 '2Y' : 2.948 , 
 '3Y' : 2.934, 
 '5Y' : 2.983, 
 '7Y' : 3.036, 
 '10Y': 3.11,  
 '12Y': 3.195,  
 '15Y': 3.2439, 
 '20Y': 3.246, 
 '25Y': 3.186,
 '30Y': 3.115 }
 
liabilities_payment_schedule = {
 '1Y': 240716648,   '2Y': 249313671,  '3Y': 260776369,  '4Y': 275104740,   
 '5Y': 292298787,   '6Y': 312358507,  '7Y': 335283902,  '8Y': 361074972, 
 '9Y': 384000367,  '10Y': 406925762, '11Y': 395463064, '12Y': 418388459,
 '13Y': 444179529, '14Y': 472836273, '15Y': 504358691, '16Y': 535881109, 
 '17Y': 561672178, '18Y': 578866225, '19Y': 587463248, '20Y': 590328922,
 '21Y': 584597573, '22Y': 590328922, '23Y': 584597573, '24Y': 573134876,
 '25Y': 555940830, '26Y': 533015435, '27Y': 504358691, '28Y': 469970598, 
 '29Y': 429851157, '30Y': 389731716 } 

In [4]:
# create a solver - enables calibration of curve to swap rates
solver = SwapCurveSolver(
 solver_id = 'eurswap_solver', 
 curve_date = curve_date, 
 instrument_rates = swap_rates
)

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 4.554749889445546e-16, `time`: 0.1157s


In [5]:
# construct the curve       
curve = SwapCurveBuilder.from_solver(
 curve_id = 'eurswap', 
 curve_solver = solver
)

In [6]:
# create the par -> forward Jacobian for the curve - needed to derive forward swap rates and calculate forward PV01
curve.calc_par_fwd_jacobian()

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0073s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


In [7]:
# run pca from csv file with time series data for swap rates
swap_rate_pca = SwapRatePCAEstimator.from_csv( 
 swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [8]:
# run mc simulation with scenarios for market/swap rates - 1000 scenarios, 1d shocks scaled to 30 day horizon
swap_rate_sim = SwapRateMCSimulator.from_csv(
  n_scenarios = 1000, 
  t_scale_factor = 30, 
  rng_seed = 1, 
  swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [9]:
# create scenarios for curves calibrated to mc simulated market/swap rates 
scenarios = SwapCurveScenarioGenerator( 
    curve_date = curve_date, 
    swap_rate_sim = swap_rate_sim, 
    swap_rate_pca = swap_rate_pca
) 

In [10]:
# calibrate the curves in each scenario of swap rates - required to generate PnL scenarios for mc CVaR calculation
# Note! For 1000 scenarios, this takes approx. 5-6 mins. 
scenarios.build_mc_scenario_curves( curve_id = 'eurswap' )

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.13574640726366e-16, `time`: 0.0916s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 6.50531716466949e-16, `time`: 0.0934s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 5.137182868220651e-16, `time`: 0.1011s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 1.6703745712172888e-16, `time`: 0.0914s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 9.661113896766674e-16, `time`: 0.0935s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.4509528820842696e-16, `time`: 0.0917s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 7.586365078553314e-16, `time`: 0.0958s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.474079273925337e-16, `time`: 0.0923s
SUCCESS: `func_tol` reached after 7 iterations (levenber

In [14]:
# create 30Y liability position 
lby_pos = LiabilityPosition(
  pos_id = 'liabilities 30Y', 
  pos_grp = 'liabilities', 
  currency = 'eur',   
  effective_dt = curve_date, 
  payment_schedule = liabilities_payment_schedule, 
  position_curve_builder = curve 
)

In [17]:
# create an instance of LiabilityHedgeConstructor - the object which generates hedges for the liability position 
lby_hedge = LiabilityHedgeConstructor(
  liability_position = lby_pos, 
  liability_curve = curve, 
  liquid_irs_tenors = ['2Y','5Y','10Y','30Y'], # this argument specifies tenors which are considered as liquid and are used to construct a proxy hedge
  swap_rate_cov_mat = scenarios.hist_cov_mat, 
  hedge_dt = curve_date )

In [18]:
# create hedged portfolio - with full benchmark hedge 
lby_prtfl_full_hedge = lby_hedge.create_hedged_portfolio(
    liability_position = lby_hedge.liability_position, 
    hedge_positions = lby_hedge.benchmark_hedge_positions, 
    effective_dt = curve_date, 
    prtfl_id = 'liabilities 30Y (benchmark hedge)' )

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0059s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0044s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0043s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0066s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0060s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0043s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0041s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0048s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0089s


In [19]:
# create hedged portfolio - with proxy hedge 
lby_prtfl_proxy_hedge = lby_hedge.create_hedged_portfolio(
    liability_position = lby_hedge.liability_position, 
    hedge_positions = lby_hedge.liquid_proxy_hedge_positions, 
    effective_dt = curve_date, 
    prtfl_id = 'liabilities 30Y (proxy hedge)' )

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0040s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s


In [20]:
# portfolio positions and risk sens.
lby_prtfl_proxy_hedge.positions
lby_prtfl_full_hedge.positions
  
lby_prtfl_proxy_hedge.spot_delta
lby_prtfl_full_hedge.spot_delta  

lby_prtfl_proxy_hedge.spot_delta
lby_prtfl_full_hedge.spot_delta  

lby_prtfl_proxy_hedge.fwd_delta
lby_prtfl_full_hedge.fwd_delta  

lby_prtfl_proxy_hedge.spot_gamma
lby_prtfl_full_hedge.spot_gamma  

,Date,Portfolio ID,Position Grp,Position ID,Sensitivity,Ccy,Curve,Total,2Y,3Y,5Y,7Y,10Y,12Y,15Y,20Y,25Y,30Y
0,2026-03-31,liabilities 30Y (benchmark hedge),liabilities,liabilities 30Y,Gamma,EUR,eurswap,-23594.033721,-142.889546,-185.621546,-418.390574,-906.97869,-1390.806697,-1097.446135,-3803.580942,-5927.500489,-6972.118421,-2748.700681
1,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 2Y ATM Payer @ 2.948%,Gamma,EUR,eurswap,-2.293829,-2.624368,0.38097,-0.06918,0.022955,-0.006553,0.002768,-0.000494,0.000098,-0.000031,0.000006
2,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 3Y ATM Receiver @ 2.934%,Gamma,EUR,eurswap,3.522969,1.12807,2.354237,0.055798,-0.018538,0.005303,-0.002243,0.000401,-0.00008,0.000025,-0.000005
3,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 5Y ATM Receiver @ 2.983%,Gamma,EUR,eurswap,10.359006,1.116888,1.489399,7.971693,-0.268212,0.076775,-0.032494,0.00582,-0.001158,0.000369,-0.000074
4,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 7Y ATM Receiver @ 3.036%,Gamma,EUR,eurswap,114.957352,6.815838,7.547314,22.825141,78.59453,-1.28538,0.542322,-0.096666,0.019085,-0.006039,0.001206
5,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 10Y ATM Receiver @ 3.11%,Gamma,EUR,eurswap,356.829803,10.083257,12.94102,28.586123,62.399801,253.748062,-12.891589,2.303887,-0.456749,0.145047,-0.029056
6,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 12Y ATM Receiver @ 3.195%,Gamma,EUR,eurswap,329.148219,6.447915,8.164769,18.588572,38.453233,60.128028,197.868196,-0.589318,0.116232,-0.036744,0.007336
7,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 15Y ATM Receiver @ 3.244%,Gamma,EUR,eurswap,2302.566594,28.343503,36.279775,80.487506,174.433141,228.558065,313.494197,1455.695171,-19.756584,6.296404,-1.264584
8,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 20Y ATM Receiver @ 3.246%,Gamma,EUR,eurswap,5794.935334,38.688628,48.923598,111.370281,230.601959,359.457999,257.673387,1010.197829,3838.815367,-126.147856,25.35414
9,2026-03-31,liabilities 30Y (benchmark hedge),Benchmark IRS Hedge,EURIBOR6M 25Y ATM Receiver @ 3.186%,Gamma,EUR,eurswap,9715.926267,39.543163,50.109425,113.274597,237.617342,353.304605,313.159454,906.964445,1723.127962,6069.567768,-90.742495


In [21]:
# simulate mc pnl for hedged portfolios - with 1000 scenarios - takes approx. 8 mins for 2 portfolio
print("start at: ", datetime.now())
lby_prtfl_proxy_hedge.simulate_pnl(mc_curve_scenarios = scenarios)
lby_prtfl_full_hedge.simulate_pnl(mc_curve_scenarios = scenarios)
print("end at: ", datetime.now())

start at:  2026-05-05 20:39:40.101711
end at:  2026-05-05 20:44:13.891218


In [23]:
# CVaR for all portfolios  
lby_prtfl_proxy_hedge.calc_cvar(0.95)
lby_prtfl_full_hedge.calc_cvar(0.95)
  
lby_prtfl_proxy_hedge.prtfl_cvar
lby_prtfl_full_hedge.prtfl_cvar
  
lby_prtfl_proxy_hedge.pos_grp_cvar
lby_prtfl_full_hedge.pos_grp_cvar
    
lby_prtfl_proxy_hedge.pos_cvar
lby_prtfl_full_hedge.pos_cvar  
  
lby_prtfl_proxy_hedge.pv01_delta_cvar_attr
lby_prtfl_full_hedge.pv01_delta_cvar_attr

lby_prtfl_proxy_hedge.sim_pos_pnl
lby_prtfl_full_hedge.sim_pos_pnl

lby_prtfl_proxy_hedge.sim_pos_grp_pnl
lby_prtfl_full_hedge.sim_pos_grp_pnl

lby_prtfl_proxy_hedge.sim_prtfl_pnl
lby_prtfl_full_hedge.sim_prtfl_pnl

,Date,Portfolio ID,Time Horizon (Days),Ccy,Curve,Scenario,PnL
0,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,1,7127.534113
1,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,2,16709.442240
2,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,3,9013.384295
3,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,4,24338.410921
4,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,5,7996.496772
...,...,...,...,...,...,...,...
995,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,996,40944.185321
996,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,997,502458.870232
997,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,998,127216.572892
998,2026-03-31,liabilities 30Y (benchmark hedge),30,EUR,eurswap,999,13069.120325
